# Train YOLO Face Detection Model in Google Colab

This notebook trains a custom YOLOv8 face detection model using the EdjeElectronics methodology.

**Key Features:**
- Uses Google Colab's free GPU
- Trains on custom face dataset
- Exports model for Raspberry Pi deployment
- Real-time inference on video/camera

**Original Guide:** https://github.com/EdjeElectronics/Train-and-Deploy-YOLO-Models

**Runtime:** 2-4 hours on Colab GPU (T4 or V100)

## 1. Import Required Libraries

Import necessary libraries for training, including Ultralytics YOLO, OpenCV, and data processing tools.

In [ ]:
# Install required packages
!pip install -q ultralytics opencv-python numpy pyyaml matplotlib pillow tqdm

# Import libraries
import os
import sys
import subprocess
from pathlib import Path
import json
import random
import shutil

# Deep learning & vision
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import yaml

# YOLO
from ultralytics import YOLO

# Data processing
from tqdm import tqdm

print("✓ All libraries imported successfully!")
print(f"Python version: {sys.version}")
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Download and Prepare YOLO Model

Set up the environment and download the pre-trained YOLOv8 model weights.

In [ ]:
# Set up working directories
work_dir = Path('/content/face_detection')
work_dir.mkdir(exist_ok=True)
os.chdir(work_dir)

print(f"Working directory: {work_dir}")
print(f"Available GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## 3. Load Pre-trained YOLO Model

Load a pre-trained YOLO model that will be fine-tuned for face detection.

In [ ]:
# Load pre-trained model
# Options: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large), yolov8x (xlarge)
# Use nano (n) for fastest training and deployment on Raspberry Pi
# Use small (s) or medium (m) for better accuracy

model_size = 'n'  # Change to 's' or 'm' for better accuracy but slower inference
model = YOLO(f'yolov8{model_size}.pt')

print(f"✓ Loaded YOLOv8{model_size} model")
print(f"Model parameters: {sum(p.numel() for p in model.model.parameters()):,}")

## 4. Prepare Dataset for Face Detection

Upload and organize your face detection dataset in YOLO format.

**Dataset Structure Required:**
```
face_detection_dataset/
├── images/
│   ├── train/
│   │   ├── image1.jpg
│   │   ├── image2.jpg
│   │   └── ...
│   └── val/
│       ├── val_image1.jpg
│       └── ...
├── labels/
│   ├── train/
│   │   ├── image1.txt  (YOLO format: 0 cx cy w h)
│   │   ├── image2.txt
│   │   └── ...
│   └── val/
│       ├── val_image1.txt
│       └── ...
├── data.yaml
└── classes.txt
```

### 4a. Option 1: Upload Dataset via Colab Files

Click the folder icon on the left sidebar → "Upload to session storage" → select `data.zip`

Then run the cell below to extract it:

In [ ]:
# Extract uploaded data.zip
import zipfile

if os.path.exists('/content/data.zip'):
    print("Found data.zip, extracting...")
    with zipfile.ZipFile('/content/data.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/face_detection')
    print("✓ Dataset extracted")
else:
    print("data.zip not found. Please upload it using the Colab file uploader.")

### 4b. Option 2: Download from Google Drive

Uncomment and run this to mount Google Drive and copy your dataset:

In [ ]:
# Uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/gdrive')
# !cp /content/gdrive/MyDrive/path/to/data.zip /content/
# !unzip -q /content/data.zip -d /content/face_detection

In [ ]:
# Verify and display dataset structure
import os
from pathlib import Path

data_dir = Path('/content/face_detection')

# Find data.yaml
data_yaml_paths = list(data_dir.rglob('data.yaml'))
if data_yaml_paths:
    data_yaml_path = data_yaml_paths[0]
    print(f"✓ Found data.yaml at: {data_yaml_path}")
    
    # Display content
    with open(data_yaml_path) as f:
        print("\ndata.yaml content:")
        print(f.read())
else:
    print("Warning: data.yaml not found")

# Count images
train_images = len(list(data_dir.glob('images/train/*')))
val_images = len(list(data_dir.glob('images/val/*')))
print(f"\n✓ Dataset summary:")
print(f"  Training images: {train_images}")
print(f"  Validation images: {val_images}")

## 5. Configure YOLO Training Parameters

Set up hyperparameters for training the face detection model.

In [ ]:
# Training configuration
config = {
    'data': str(data_yaml_path),           # Path to data.yaml
    'epochs': 100,                         # Number of training epochs (50-100 recommended)
    'batch': 16,                           # Batch size (increase if GPU memory allows)
    'imgsz': 640,                          # Input image size
    'device': 0,                           # GPU device (0 for first GPU)
    'patience': 20,                        # Early stopping patience
    'save': True,                          # Save checkpoints
    'augment': True,                       # Data augmentation
    'hsv_h': 0.015,                        # HSV-Hue augmentation
    'hsv_s': 0.7,                          # HSV-Saturation augmentation
    'hsv_v': 0.4,                          # HSV-Value augmentation
    'degrees': 10.0,                       # Rotation degrees
    'translate': 0.1,                      # Translation
    'scale': 0.5,                          # Scale
    'flipud': 0.5,                         # Flip upside-down
    'fliplr': 0.5,                         # Flip left-right
    'mosaic': 1.0,                         # Mosaic augmentation
    'mixup': 0.0,                          # Mixup augmentation
    'lr0': 0.01,                           # Initial learning rate
    'lrf': 0.01,                           # Final learning rate
    'momentum': 0.937,                     # Momentum
    'weight_decay': 0.0005,                # Weight decay
    'warmup_epochs': 3.0,                  # Warmup epochs
    'warmup_momentum': 0.8,                # Warmup momentum
    'box': 7.5,                            # Box loss weight
    'cls': 0.5,                            # Class loss weight
    'dfl': 1.5,                            # DFL loss weight
    'name': 'face_detector_v1'             # Experiment name
}

print("✓ Training configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 6. Train the Face Detection Model

Start training the YOLO model on the face detection dataset. This will take 2-4 hours on Colab GPU.

In [ ]:
print("Starting YOLO face detection model training...")
print("This will take 2-4 hours on Colab GPU (T4) or 8-12 hours on CPU")
print("=" * 60)

# Train the model
results = model.train(**config)

print("=" * 60)
print("✓ Training completed!")
print(f"Best model saved to: {results.save_dir}/weights/best.pt")

## 7. Evaluate Model Performance

View training results and metrics from the trained model.

In [ ]:
# Display training results graph
from PIL import Image
import matplotlib.pyplot as plt

# Find results image
results_img_path = None
for root, dirs, files in os.walk('/content/face_detection/runs'):
    if 'results.png' in files:
        results_img_path = os.path.join(root, 'results.png')
        break

if results_img_path:
    print(f"Loading training results from: {results_img_path}")
    img = Image.open(results_img_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Results image not found")

In [ ]:
# Print training metrics
import csv

# Find results CSV
results_csv_path = None
for root, dirs, files in os.walk('/content/face_detection/runs'):
    if 'results.csv' in files:
        results_csv_path = os.path.join(root, 'results.csv')
        break

if results_csv_path:
    print("Training Metrics (Last 5 Epochs):")
    print("=" * 100)
    with open(results_csv_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        
        # Print header
        if rows:
            headers = list(rows[0].keys())
            print(",".join(headers[:6]))  # Print first 6 columns
            
            # Print last 5 rows
            for row in rows[-5:]:
                values = [row[h][:8] for h in headers[:6]]
                print(",".join(values))

## 8. Run Inference on Test Images

Test the trained model on sample images from the validation set.

In [ ]:
# Load best trained model
from ultralytics import YOLO

best_model_path = '/content/face_detection/runs/detect/face_detector_v1/weights/best.pt'
trained_model = YOLO(best_model_path)

print(f"✓ Loaded trained model from: {best_model_path}")

# Run inference on validation images
val_images_dir = '/content/face_detection/images/val'
results_list = trained_model.predict(
    source=val_images_dir,
    conf=0.5,  # Confidence threshold
    device=0,
    save=True,
    project='/content/face_detection/runs/detect',
    name='inference_results'
)

print(f"✓ Inference completed on {len(results_list)} validation images")

In [ ]:
# Display sample inference results
from PIL import Image
import matplotlib.pyplot as plt
import glob

# Find inference result images
result_images = glob.glob('/content/face_detection/runs/detect/inference_results/*.jpg')[:6]

print(f"Displaying {len(result_images)} inference results:\n")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, img_path in enumerate(result_images):
    img = Image.open(img_path)
    axes[idx].imshow(img)
    axes[idx].set_title(Path(img_path).stem)
    axes[idx].axis('off')

# Hide empty subplots
for idx in range(len(result_images), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 9. Deploy Model for Real-time Detection

Export the trained model and prepare it for deployment on Raspberry Pi.

### 9a. Export to ONNX Format (for Raspberry Pi)

In [ ]:
# Export model to ONNX format
print("Exporting model to ONNX format for Raspberry Pi...")

# Export best model
exported_model_path = trained_model.export(format='onnx', imgsz=640)
print(f"✓ Model exported to: {exported_model_path}")

# Verify export
if os.path.exists(exported_model_path):
    file_size_mb = os.path.getsize(exported_model_path) / (1024 * 1024)
    print(f"  File size: {file_size_mb:.1f} MB")
else:
    print("  Error: Export failed")

### 9b. Download Model Files

In [ ]:
# Create zip file with model and training results
import zipfile
import shutil

# Create deployment package
deployment_dir = Path('/content/face_detection/my_face_model')
deployment_dir.mkdir(exist_ok=True)

# Copy model files
print("Creating deployment package...")

# Copy best.pt and best.onnx
shutil.copy(
    '/content/face_detection/runs/detect/face_detector_v1/weights/best.pt',
    deployment_dir / 'best.pt'
)
shutil.copy(
    exported_model_path,
    deployment_dir / 'best.onnx'
)

# Copy results
shutil.copy(
    '/content/face_detection/runs/detect/face_detector_v1/results.csv',
    deployment_dir / 'results.csv'
)
shutil.copy(
    '/content/face_detection/runs/detect/face_detector_v1/results.png',
    deployment_dir / 'results.png'
)

# Create zip
zip_path = '/content/my_face_model.zip'
shutil.make_archive('/content/my_face_model', 'zip', deployment_dir.parent, deployment_dir.name)

print(f"✓ Deployment package created: {zip_path}")
print(f"  Size: {os.path.getsize(zip_path) / (1024 * 1024):.1f} MB")

In [ ]:
# Download model files
from google.colab import files

print("\nDownloading deployment package...")
print("This may take a minute...\n")

# Download zip
files.download('/content/my_face_model.zip')

print("✓ Download complete!")
print("\nNext steps:")
print("1. Extract my_face_model.zip on your computer")
print("2. Upload best.onnx to Raspberry Pi")
print("3. Run with yolodesk on Pi: python yolodesk live --model best.onnx --source https://192.168.1.111:8002/mjpeg --insecure")

### 9c. Deploy to Raspberry Pi

After downloading, follow these steps on your Raspberry Pi:

```bash
# 1. Transfer the ONNX model to your Pi
scp my_face_model/best.onnx billyp@192.168.1.111:/home/billyp/Documents/Face\ Detection/models/my_custom_face_detector.onnx

# 2. On the Pi, run inference
cd ~/Documents/Face\ Detection/yolo_face
source .venv/bin/activate

# 3. Run with your custom model
python yolodesk live \
    --model models/my_custom_face_detector.onnx \
    --source "https://192.168.1.111:8002/mjpeg" \
    --insecure \
    --conf 0.6 \
    --serve --serve-port 8003

# 4. View results in browser
# http://192.168.1.111:8003/
```

**Performance Expectations:**
- **YOLOv8n (nano):** ~8-10 FPS on Raspberry Pi 5
- **YOLOv8s (small):** ~4-6 FPS on Raspberry Pi 5
- **YOLOv8m (medium):** ~2-3 FPS on Raspberry Pi 5